In [13]:
import torch
from torch import Tensor


def extend_attention_mask(
    attention_mask: Tensor,
    prefix_length: int,
) -> Tensor:
    """Tạo Extended Attention Mask bằng cách nối Prefix Mask (toàn 1) vào trước Caption Attention Mask.

    Args:
        attention_mask (Tensor): Mask của caption [B, L], giá trị 1 cho token thật và 0 cho padding.
        prefix_length (int): Số lượng visual prefix tokens (P).

    Returns:
        Tensor: Extended attention mask có shape [B, P + L], giữ nguyên dtype và device của input.
    """
    if attention_mask.ndim != 2:
        raise ValueError(
            f"attention_mask phải có shape [B, L], nhận được: {tuple(attention_mask.shape)}"
        )

    if (
        isinstance(prefix_length, bool)
        or not isinstance(prefix_length, int)
        or prefix_length <= 0
    ):
        raise ValueError(
            f"prefix_length phải là số nguyên dương > 0, nhận được: {prefix_length}"
        )

    batch_size = attention_mask.size(0)

    # Prefix Mask = 1 cho toàn bộ P vị trí prefix [B, P]
    prefix_mask = torch.ones(
        (batch_size, prefix_length),
        dtype=attention_mask.dtype,
        device=attention_mask.device,
    )

    # Nối Prefix Mask [B, P] và Caption Attention Mask [B, L] -> [B, P + L]
    return torch.cat([prefix_mask, attention_mask], dim=1)

In [15]:
# Sanity check & Báo cáo chi tiết cho Extended Attention Mask
batch_size = 32
caption_length = 20
prefix_length = 10

# 1. Giả lập caption attention mask (với vài vị trí padding)
test_caption_mask = torch.ones(batch_size, caption_length, dtype=torch.long)
test_caption_mask[:, 15:] = 0  # Giả lập 5 token cuối là padding

# 2. Thực thi hàm
extended_mask = extend_attention_mask(test_caption_mask, prefix_length=prefix_length)

# 3. In Báo cáo Chi tiết
print(f"Input Shape (Caption Mask)     : {tuple(test_caption_mask.shape)}")
print(f"Prefix Length                  : {prefix_length}")
print(f"Output Shape (Extended Mask)   : {tuple(extended_mask.shape)}")
print(f"Prefix Mask Portion Check      : {extended_mask[0, :prefix_length].tolist()}")
print(f"Caption Mask Portion Check     : {extended_mask[0, prefix_length:].tolist()}")

# 4. Kiểm tra Assertion
expected_shape = (batch_size, prefix_length + caption_length)
assert extended_mask.shape == expected_shape, f"ERR: Shape không bằng {expected_shape}"
assert (extended_mask[:, :prefix_length] == 1).all(), "ERR: Prefix mask phải toàn giá trị 1"
assert torch.equal(extended_mask[:, prefix_length:], test_caption_mask), "ERR: Phần caption mask bị sai"

print("PASS")

Input Shape (Caption Mask)     : (32, 20)
Prefix Length                  : 10
Output Shape (Extended Mask)   : (32, 30)
Prefix Mask Portion Check      : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Caption Mask Portion Check     : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
PASS
